In [4]:
# 1) Load model + tokenizer (frozen)
import torch
from transformers import BertModel, BertTokenizer
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "Rostlab/prot_bert"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
bert_model = BertModel.from_pretrained(MODEL_NAME).to(device).eval()
for p in bert_model.parameters():
    p.requires_grad = False


def prep_sequence(seq: str) -> str:
    seq = seq.upper().replace('U', 'X').replace('Z', 'X').replace('O', 'X').replace('B', 'X')
    return " ".join(list(seq))


@torch.no_grad()
def embed_sequences(seqs, batch_size=16, pooling="mean"):
    all_embeddings = []
    prepped = [prep_sequence(s) for s in seqs]
    for i in range(0, len(prepped), batch_size):
        batch = prepped[i:i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=64)
        enc = {k: v.to(device) for k, v in enc.items()}
        out = bert_model(**enc)
        last_hidden = out.last_hidden_state
        if pooling == "cls":
            pooled = last_hidden[:, 0, :]
        else:
            mask = enc["attention_mask"].unsqueeze(-1).float()
            summed = (last_hidden * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            pooled = summed / counts
        all_embeddings.append(pooled.cpu().numpy())
    return np.vstack(all_embeddings)


# 2) Load your data (adjust path to match your Kaggle input)
file_path = "/kaggle/input/datasets/benalayayesmine/dataset1/dataset.xlsx"
xls = pd.ExcelFile(file_path)
dfs = [pd.read_excel(xls, sheet_name=s) for s in xls.sheet_names]
data = pd.concat(dfs, ignore_index=True)
data["length"] = data["Sequence"].str.len()
print("Dataset shape:", data.shape)


# 3) THIS is the step that was skipped — actually compute embeddings
embeddings = embed_sequences(data["Sequence"].tolist(), batch_size=16, pooling="mean")
print("Embedding matrix shape:", embeddings.shape)   # should be (728, 1024)


# 4) PCA + Ridge pipeline factory
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

def make_pca_pipeline(n_components=30):
    return make_pipeline(
        StandardScaler(),
        PCA(n_components=n_components, random_state=42),
        Ridge(alpha=1.0),
    )


# 5) Length-stratified repeated CV — now `embeddings` exists
from sklearn.model_selection import RepeatedKFold, cross_validate

cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=42)
emb_cols = [f'protbert_{i}' for i in range(embeddings.shape[1])]
emb_df = pd.DataFrame(embeddings, columns=emb_cols, index=data.index)
data_full = pd.concat([data, emb_df], axis=1)

print('=== ProtBERT-embedding Ridge (with PCA), length-stratified ===')
for L, grp in data_full.groupby('length'):
    if len(grp) < 30:
        continue
    X = grp[emb_cols]
    y = grp['log(IC50)']
    n_comp = min(30, len(grp) - 20)
    reg_model = make_pca_pipeline(n_components=n_comp)
    scores = cross_validate(reg_model, X, y, cv=cv, scoring='r2')
    print(f'length={L:2d} n={len(grp):4d}  R2={scores["test_score"].mean():+.4f} '
          f'+/- {scores["test_score"].std():.4f}')

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dataset shape: (728, 3)
Embedding matrix shape: (728, 1024)
=== ProtBERT-embedding Ridge (with PCA), length-stratified ===
length= 2 n= 133  R2=-0.1759 +/- 0.2194
length= 3 n= 215  R2=-0.0473 +/- 0.1692
length= 4 n=  90  R2=-0.6297 +/- 0.6093
length= 5 n= 132  R2=-0.2589 +/- 0.3051
length= 6 n= 114  R2=-0.1209 +/- 0.2554


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

seqs_check = ['LL', 'DD', 'RR', 'GG']
check_emb = embed_sequences(seqs_check, pooling='mean')  # uses bert_model, no shadowing this time
print(pd.DataFrame(cosine_similarity(check_emb), index=seqs_check, columns=seqs_check).round(3))

       LL     DD     RR     GG
LL  1.000  0.783  0.738  0.795
DD  0.783  1.000  0.876  0.887
RR  0.738  0.876  1.000  0.801
GG  0.795  0.887  0.801  1.000
